# 🩺 Multilingual Healthcare Misinformation Detection & Verification (RAG)
### Cross-Lingual Clinical Fact-Checking in English, Hindi, and Hinglish using Dense Retrieval

This project delivers an end-to-end clinical fact-checking and Retrieval-Augmented Generation (RAG) system with three foundational components:
1. **EDA (Exploratory Data Analysis)** (3 marks): Thorough exploration of the clinical misinformation dataset (`Datensatz.csv`), label distributions, class imbalances, claim lengths, vocabulary analysis, and text statistics.
2. **Data Preprocessing** (3 marks): Noise removal, Devanagari Hindi Unicode preservation, Hinglish variant normalization, URL/mention cleaning, and claim-evidence contextualization.
3. **Model Identification** (3 marks): Comparative evaluation of candidate multilingual embedding and classification architectures (paraphrase-multilingual-MiniLM-L12-v2 vs. MuRIL vs. mBERT vs. fine-tuned Clinical models), hardware trade-offs, and FAISS indexing selection.
4. **Interactive Deployment**: Complete 10-step RAG verification pipeline with risk assessment, explainable citations, and a live Gradio web application.


## ⚙️ Step 1 — Environment Setup & Hardware Detection
Auto-detect available hardware acceleration (GPU/CUDA vs. CPU). While CPU execution is supported, GPU enables high-throughput batch vectorization.


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Hardware Acceleration Device: {device.upper()}")
if device == "cuda":
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
else:
    print("   Running on CPU (Fully supported for this model and dataset size)")


## 📦 Step 2 — Install Required Libraries
Installs core NLP, data processing, visualization, and RAG libraries:
- `sentence-transformers`: Dense multilingual semantic representations
- `faiss-cpu`: High-speed inner-product dense vector search
- `langdetect`: Language classification
- `pandas`, `numpy`, `scikit-learn`: Data structures, metrics, and evaluation
- `matplotlib`, `seaborn`: Statistical charting for Exploratory Data Analysis (EDA)
- `gradio`: Interactive clinical verification UI


In [ ]:
!pip -q install sentence-transformers faiss-cpu langdetect pandas scikit-learn gradio matplotlib seaborn

import os
import re
import unicodedata
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("✅ Core NLP, Data Science, and RAG libraries successfully imported!")


## 📂 Step 3 — Load Clinical Dataset (`Datensatz.csv` / `evidence_dataset.csv`)
Loads the primary clinical evidence dataset synthesized from *Medizin Transparent* clinical trials and systematic reviews.
- `id`: Unique record identifier
- `claim_topic`: Clinical topic / investigated medical claim
- `evidence_text`: Verified medical consensus & trial findings
- `stance`: `supports` (Label 0), `uncertain` (Label 1), or `refutes` (Label 2)
- `source`: Reputable institutional publisher (*Medizin Transparent / Cochrane*)
- `url`: Direct source link for transparency


In [ ]:
DATASET_PATH = "data/evidence_dataset.csv"
GITHUB_RAW_URL = "https://raw.githubusercontent.com/ay10101/multilingual-healthcare-rag/main/data/evidence_dataset.csv"

if os.path.exists(DATASET_PATH):
    evidence_df = pd.read_csv(DATASET_PATH)
else:
    print("Fetching evidence dataset directly from GitHub repository...")
    evidence_df = pd.read_csv(GITHUB_RAW_URL)

print(f"✅ Loaded {len(evidence_df)} clinical records across {evidence_df['claim_topic'].nunique()} topics.")
print(f"Columns: {list(evidence_df.columns)}")
evidence_df.head(3)


---
# 📊 Section 1: Exploratory Data Analysis (EDA) — [3 Marks]

In this section, we conduct a systematic exploratory analysis of the clinical dataset:
1. **Class Distribution & Imbalance**: Quantifying the frequency of `supports`, `uncertain`, and `refutes` labels.
2. **Text Length & Token Distribution**: Analyzing word counts and character lengths for claims vs. clinical evidence.
3. **Missing Value & Data Quality Audit**: Identifying completeness of URLs, sources, and text fields.
4. **Top Clinical Topics**: Discovering frequent medical themes (COVID-19, cancer, vaccines, diet, pain).


In [ ]:
# 1. Dataset Shape and Missing Values Audit
print("--- 1. Data Integrity & Null Counts ---")
print(f"Total Records: {len(evidence_df)}")
null_summary = evidence_df.isnull().sum()
print(null_summary)

# 2. Class / Stance Distribution
print("\n--- 2. Stance Distribution ---")
stance_counts = evidence_df['stance'].value_counts()
stance_pct = evidence_df['stance'].value_counts(normalize=True) * 100
for stance, count in stance_counts.items():
    print(f"  • {stance.upper():<10}: {count} records ({stance_pct[stance]:.2f}%)")

# 3. Text Length Metrics
evidence_df['claim_word_count'] = evidence_df['claim_topic'].apply(lambda x: len(str(x).split()))
evidence_df['evidence_word_count'] = evidence_df['evidence_text'].apply(lambda x: len(str(x).split()))

print("\n--- 3. Word Count Statistics ---")
print(evidence_df[['claim_word_count', 'evidence_word_count']].describe().round(1))


In [ ]:
# Visualization: EDA Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Chart 1: Stance Distribution (Bar chart)
colors = {"uncertain": "#e67e22", "supports": "#27ae60", "refutes": "#c0392b"}
bar_colors = [colors.get(s, "#3498db") for s in stance_counts.index]
axes[0].bar(stance_counts.index, stance_counts.values, color=bar_colors, edgecolor="black", alpha=0.85)
axes[0].set_title("Clinical Evidence Stance Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Stance")
axes[0].set_ylabel("Count")
for i, (stance, val) in enumerate(stance_counts.items()):
    axes[0].text(i, val + 8, f"{val} ({stance_pct[stance]:.1f}%)", ha="center", fontweight="bold")

# Chart 2: Evidence Text Word Count Distribution (Histogram)
sns.histplot(evidence_df['evidence_word_count'], bins=30, kde=True, ax=axes[1], color="#2980b9")
axes[1].set_title("Evidence Text Word Count Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")

# Chart 3: Claim Word Count by Stance (Boxplot)
sns.boxplot(data=evidence_df, x="stance", y="claim_word_count", palette=colors, ax=axes[2])
axes[2].set_title("Claim Word Count by Stance Category", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Stance")
axes[2].set_ylabel("Words per Claim")

plt.tight_layout()
plt.show()

# Key EDA Insights
print("""
📌 Key EDA Insights:
1. Class Imbalance: ~56% of investigated medical claims have 'uncertain' (insufficient scientific evidence) status,
   accurately reflecting real-world clinical research where unproven therapies lack rigorous randomized controlled trials.
2. Verified Stances: 27% of claims are corroborated ('supports') and ~17% are directly debunked ('refutes').
3. Evidence Granularity: Evidence findings average ~60-80 words, providing rich semantic context for dense neural retrieval.
""")


---
# 🧹 Section 2: Data Preprocessing Pipeline — [3 Marks]
## Step 4 — Text Preprocessing
Data preprocessing is vital for cross-lingual misinformation detection across English, Hindi, and Hinglish:
1. **Case Normalization & Noise Cleaning**: Strips URLs, `@mentions`, `#hashtags`, and rogue control characters.
2. **Unicode Preservation**: Preserves Devanagari Hindi characters (`\u0900-\u097F`) alongside Latin alphanumerics.
3. **Hinglish Variant Normalization**: Maps informal Romanized Hindi spellings (`thik` ➔ `theek`, `dawa` ➔ `dawai`, `ilaaj` ➔ `ilaj`) into canonical representations.
4. **Claim-Evidence Contextualization**: Pre-formats evidence entries as `Claim: ... | Finding: ...` to maximize semantic alignment during dense neural search.


In [ ]:
# Hinglish spelling normalization dictionary
HINGLISH_NORMALIZATION_MAP = {
    r"\bthik\b": "theek",
    r"\bteek\b": "theek",
    r"\bilaaj\b": "ilaj",
    r"\bdawa\b": "dawai",
    r"\bdawaai\b": "dawai",
    r"\bdawayi\b": "dawai",
    r"\bh\b": "hai",
    r"\bkartey\b": "karte",
    r"\bkarti\b": "karta",
    r"\bpeeney\b": "peena",
    r"\bpeene\b": "peena",
    r"\bkhatm\b": "khatam",
    r"\bmadhumeh\b": "diabetes",
    r"\bbimaari\b": "bimari"
}

def preprocess_text(text: str) -> str:
    """
    Cleans and normalizes query text:
    - Lowercases and normalizes Unicode (NFKC)
    - Removes URLs, mentions, hashtags, and noisy symbols
    - Preserves Devanagari Hindi characters (\u0900-\u097F)
    - Normalizes Hinglish colloquial variants
    """
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[@#]\S+", " ", text)
    text = re.sub(r"[^\w\s\u0900-\u097F\.,?!-]", " ", text)
    for pattern, replacement in HINGLISH_NORMALIZATION_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Verification of Preprocessing Pipeline on Diverse Test Inputs
sample_test_inputs = [
    "Haldi doodh cancer ko thik karta h https://fake-cure.org #cure",
    "Vaccines autism cause karte hain @antivax",
    "हल्दी दूध कैंसर ठीक करता है! Check http://test.com",
    "Smoking causes lung cancer.",
    "Antibiotics cold ko cure kar dete hain"
]

print("--- Preprocessing Pipeline Verification ---")
for raw in sample_test_inputs:
    print(f"Raw Input : {raw}")
    print(f"Cleaned   : {preprocess_text(raw)}\n")


## Step 5 — Language Identification
We implement a hybrid language detector:
- **Hindi**: Identified if Devanagari script characters (`\u0900-\u097F`) are present.
- **Hinglish**: Identified if Latin script contains common Romanized Hindi markers (`hai`, `kya`, `karta`, `ke`, `mein`, `dawai`, `ko`, `se`, `nahi`, etc.).
- **English**: Fallback via `langdetect` or default.


In [ ]:
HINGLISH_KEYWORDS = {
    "hai", "hain", "kya", "karta", "karti", "karte", "ke", "ki", "ka", "ko",
    "mein", "me", "se", "par", "bhi", "nahi", "nahin", "dawai", "dawa",
    "theek", "thik", "bimari", "ilaj", "kare", "karo", "peena", "peene",
    "khatam", "hota", "hoti", "hote", "doodh", "haldi", "ghee", "paani"
}

def detect_language(text: str) -> str:
    if not text or not isinstance(text, str):
        return "English"
    if re.search(r"[\u0900-\u097F]", text):
        return "Hindi"
    clean_words = set(re.findall(r"\b[a-z]+\b", text.lower()))
    if clean_words.intersection(HINGLISH_KEYWORDS):
        return "Hinglish"
    try:
        from langdetect import detect
        detected = detect(text)
        return "Hindi" if detected == "hi" else "English"
    except Exception:
        return "English"



## Step 6 — Health-Related Claim Classification
Before passing queries to vector search, we filter out non-health claims.
- **Health keywords**: Clinical terminology (cancer, vaccine, covid, masks, cbd, osteoarthritis, fever, antibiotic, etc.) plus Hindi/Hinglish clinical terms (bimari, ilaj, dawai, haldi, dard, sehat).
- *Note*: As planned future work, this rule-based classifier can be upgraded to a fine-tuned Clinical BioBERT / MuRIL model.


In [ ]:
HEALTH_KEYWORDS = {
    "cancer", "vaccine", "vaccines", "vaccination", "medicine", "medicines",
    "doctor", "doctors", "hospital", "hospitals", "disease", "diseases",
    "diabetes", "fever", "antibiotic", "antibiotics", "covid", "covid-19",
    "health", "treatment", "cure", "cures", "curing", "virus", "viral",
    "cough", "infection", "infections", "pregnancy", "pregnant", "blood", "heart",
    "smoking", "tobacco", "cigarette", "lungs", "lung", "hypertension",
    "stroke", "tumor", "chemo", "chemotherapy", "dosage", "drug", "drugs",
    "pain", "autism", "immune", "immunity", "handwash", "handwashing", "soap",
    "hygiene", "sanitize", "sanitizer", "mask", "masks", "corona", "cbd",
    "cannabis", "cannabidiol", "headache", "headaches", "migraine", "migraines",
    "osteoarthritis", "arthritis", "joint", "joints", "knee", "herbal", "therapy",
    "paxlovid", "allergy", "allergies", "syndrome", "disorder", "symptom", "symptoms",
    "clinical", "patient", "patients", "supplement", "supplements", "vitamin", "vitamins",
    "cold", "colds", "sore", "sores", "cold sores", "cold sore", "herpes", "lysine", "blister", "blisters",
    "prevent", "preventive", "prevention", "relieve", "alleviate", "heal", "healing",
    "remedy", "remedies", "dementia", "alzheimer", "prostate", "skin", "muscle", "back pain",
    "cystitis", "urinary", "kidney", "liver", "ulcer", "gut", "stomach", "acne", "asthma",
    "bimari", "bimaari", "ilaj", "ilaaj", "dawa", "dawai", "haldi", "doodh",
    "dard", "sehat", "aspatal", "khansi", "bukhar", "sugar", "chot",
    "कैंसर", "वैक्सीन", "टीका", "दवा", "इलाज", "बीमारी", "मधुमेह", "बुखार"
}

def is_health_related(text: str) -> bool:
    if not text or not isinstance(text, str):
        return False
    tokens = set(re.findall(r"[\w\u0900-\u097F]+", text.lower()))
    if tokens.intersection(HEALTH_KEYWORDS):
        return True
    text_lower = text.lower()
    return any(kw in text_lower for kw in HEALTH_KEYWORDS)

print("--- Language & Health Filter Tests ---")
print("1. 'Haldi doodh cancer theek karta hai' -> Lang:", detect_language("Haldi doodh cancer theek karta hai"), "| Health:", is_health_related("Haldi doodh cancer theek karta hai"))
print("2. 'Can masks reduce corona infections?'  -> Lang:", detect_language("Can masks reduce corona infections?"), "| Health:", is_health_related("Can masks reduce corona infections?"))
print("3. 'India won the cricket match'         -> Lang:", detect_language("India won the cricket match"), "| Health:", is_health_related("India won the cricket match"))


---
# 🧠 Section 3: Model Identification & Architecture Selection — [3 Marks]

Choosing the right model architecture for cross-lingual healthcare fact-checking requires balancing **semantic alignment**, **latency**, **hardware efficiency**, and **multilingual representation**:

### Architectural Evaluation Matrix

| Model Candidate | Architecture | Languages | Parameters | Cross-Lingual Zero-Shot Transfer | Retrieval Speed (FAISS) | Verdict |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **`paraphrase-multilingual-MiniLM-L12-v2`** *(Selected)* | SentenceTransformer (MiniLM) | 50+ languages | ~118M | **High** (Maps Hinglish/Hindi/English to aligned space) | **Ultra-Fast (< 5ms)** | **Optimal Selection** |
| `google/muril-base-cased` | BERT-base (MuRIL) | 17 Indian Languages + English | ~236M | High for Indian languages, lower out-of-the-box sentence pooling | Slower; requires mean pooling layer | Viable Future Upgrade |
| `bert-base-multilingual-cased` (mBERT) | BERT-base | 104 languages | ~178M | Moderate (Not explicitly contrastive-trained for sentences) | Moderate | Sub-optimal for semantic retrieval |
| `BioBERT / ClinicalBERT` | Domain-specific BERT | English only | ~110M | Fails on Hindi & Hinglish queries | Fast | English-only limitation |

### Why `paraphrase-multilingual-MiniLM-L12-v2` is Chosen:
1. **Sentence-Level Contrastive Alignment**: Pre-trained specifically on parallel multilingual paraphrase corpora, enabling Hinglish claims (*“Haldi doodh cancer ko theek karta hai”*) to directly match English clinical evidence.
2. **Vector Space Efficiency**: 384-dimensional dense vectors minimize memory footprint while enabling inner-product FAISS cosine search.
3. **Low Latency & CPU/GPU Compatibility**: Executes smoothly on CPU and scales seamlessly with GPU acceleration.


In [ ]:
# Model Initialization & Architecture Summary
print("Initializing Selected Model: paraphrase-multilingual-MiniLM-L12-v2...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

print(f"✅ Model Loaded Successfully on: {device.upper()}")
print(f"   Embedding Dimension : {embed_model.get_sentence_embedding_dimension()}")
print(f"   Max Sequence Length : {embed_model.max_seq_length}")
print(f"   Tokenizer Vocabulary: {embed_model.tokenizer.vocab_size}")


## Step 7 — Build FAISS Dense Vector Index
Encodes the 750 clinical evidence texts into unit-normalized 384-dimensional embeddings and indexes them using `faiss.IndexFlatIP` (Inner Product on unit vectors equals exact Cosine Similarity).


In [ ]:
# Prevent thread conflicts on macOS
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"

print(f"Encoding {len(evidence_df)} clinical evidence texts...")
evidence_texts = evidence_df["evidence_text"].tolist()
evidence_embeddings = embed_model.encode(
    evidence_texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    device=device
)
evidence_embeddings = np.array(evidence_embeddings, dtype=np.float32)

# Build FAISS IndexFlatIP (exact cosine similarity)
embedding_dim = evidence_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(evidence_embeddings)

print(f"✅ FAISS Index successfully built with {faiss_index.ntotal} vectors of dimension {embedding_dim}!")

def retrieve_evidence(claim: str, top_k: int = 3):
    """Retrieves top-k closest clinical evidence records using FAISS inner product."""
    cleaned = preprocess_text(claim)
    q_vec = embed_model.encode([cleaned], normalize_embeddings=True, device=device)
    q_vec = np.array(q_vec, dtype=np.float32)
    scores, indices = faiss_index.search(q_vec, min(top_k, len(evidence_df)))
    
    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
        row = evidence_df.iloc[idx].to_dict()
        row["similarity_score"] = float(round(float(score), 4))
        row["rank"] = rank + 1
        results.append(row)
    return results

# Verification retrieval
sample_claim = "Does arthroscopy help with osteoarthritis of the knee joint?"
retrieved_sample = retrieve_evidence(sample_claim, top_k=2)
for r in retrieved_sample:
    print(f"Rank {r['rank']} [Similarity: {r['similarity_score']}] ({r['stance'].upper()}): {r['evidence_text'][:90]}...")


## Step 8 — Verdict Assessment & Risk Categorization
Implements transparent clinical verification logic:
- **Supported**: Closest evidence stance is `supports` and similarity $\ge 0.35$
- **Refuted**: Closest evidence stance is `refutes` and similarity $\ge 0.35$
- **Uncertain / Insufficient evidence**: Moderate similarity or conflicting findings on the same topic
- **No relevant evidence**: Low similarity score ($< 0.22$)

**Risk Level Mapping**:
- `Supported` ➔ **Low Risk**
- `Uncertain` ➔ **Medium Risk**
- `Refuted` ➔ **High Risk**
- `Refuted + Dangerous Advice` (stopping chemo, skipping insulin, avoiding vaccines) ➔ **Very High Risk**


In [ ]:
DANGEROUS_PATTERNS = [
    "stop chemo", "skip chemo", "stop insulin", "avoid vaccine", "dawai band",
    "chemo mat lo", "insulin mat lo", "teeka mat lagao", "toxic", "bleach"
]

HIGH_SIMILARITY_THRESHOLD = 0.35
MODERATE_SIMILARITY_THRESHOLD = 0.22

def assess_verdict_and_risk(claim: str, retrieved_evidence):
    if not retrieved_evidence:
        return {
            "verdict": "No relevant evidence",
            "evidence_status": "No matching clinical documents found",
            "risk": "Medium",
            "confidence": 0.0,
            "stance": "none"
        }

    top = retrieved_evidence[0]
    top_score = top.get("similarity_score", 0.0)
    top_stance = top.get("stance", "").strip().lower()
    top_topic = top.get("claim_topic", "").strip().lower()

    # Detect conflict among competitively close matches on the same topic
    has_conflict = any(
        item.get("claim_topic", "").strip().lower() == top_topic
        and item.get("stance", "").strip().lower() != top_stance
        and (top_score - item.get("similarity_score", 0.0)) <= 0.12
        and item.get("similarity_score", 0.0) >= MODERATE_SIMILARITY_THRESHOLD
        for item in retrieved_evidence[1:]
    )

    if top_score < MODERATE_SIMILARITY_THRESHOLD:
        verdict = "No relevant evidence"
        evidence_status = "No close clinical evidence found in trusted repository"
        risk = "Medium"
    elif has_conflict:
        verdict = "Uncertain / Insufficient evidence"
        evidence_status = "Conflicting clinical evidence found among top sources"
        risk = "Medium"
    elif top_score >= HIGH_SIMILARITY_THRESHOLD:
        if top_stance == "refutes":
            verdict = "Refuted"
            evidence_status = "Directly refuted by reputable clinical evidence"
            claim_lower = claim.lower()
            is_dangerous = any(pat in claim_lower for pat in DANGEROUS_PATTERNS)
            risk = "Very High" if is_dangerous else "High"
        elif top_stance == "supports":
            verdict = "Supported"
            evidence_status = "Corroborated by verified medical sources"
            risk = "Low"
        else:
            verdict = "Uncertain / Insufficient evidence"
            evidence_status = "Evidence stance inconclusive / scientific proof lacking"
            risk = "Medium"
    else:
        verdict = "Uncertain / Insufficient evidence"
        evidence_status = "Moderate similarity; insufficient clinical evidence"
        risk = "Medium"

    return {
        "verdict": verdict,
        "evidence_status": evidence_status,
        "risk": risk,
        "confidence": top_score,
        "stance": top_stance
    }


## Step 9 — Standardized Educational Explanation Generator
Adheres to medical AI compliance by generating transparent, cited, non-hallucinatory explanations accompanied by clickable citations and a medical disclaimer.


In [ ]:
def generate_explanation(original_claim, detected_language, is_health, verdict_data, retrieved_evidence):
    verdict = verdict_data["verdict"]
    risk = verdict_data["risk"]
    evidence_status = verdict_data["evidence_status"]

    if not is_health:
        summary_text = (
            "The claim was classified as **NOT health-related**. "
            "Our clinical verification engine only evaluates health, medical, or biomedical statements. "
            "No clinical retrieval was performed."
        )
        return {
            "original_claim": original_claim,
            "detected_language": detected_language,
            "is_health_related": False,
            "verdict": "Non-Health Claim",
            "evidence_status": "Not Applicable",
            "risk": "None",
            "summary_text": summary_text,
            "retrieved_evidence": [],
            "disclaimer": "This tool is for educational purposes only and does not constitute medical advice."
        }

    if verdict == "Supported":
        action_phrase = "supports"
    elif verdict == "Refuted":
        action_phrase = "refutes"
    else:
        action_phrase = "does not sufficiently address"

    summary_text = (
        f"The claim was classified as **{verdict}** based on the most relevant retrieved medical evidence.\n\n"
        f"The evidence **{action_phrase}** the claim.\n\n"
        f"- **Assessed Risk**: `{risk}`\n"
        f"- **Evidence Status**: {evidence_status}\n\n"
        f"⚠️ *This tool is for educational purposes and is not medical advice.*"
    )

    return {
        "original_claim": original_claim,
        "detected_language": detected_language,
        "is_health_related": True,
        "verdict": verdict,
        "evidence_status": evidence_status,
        "risk": risk,
        "summary_text": summary_text,
        "retrieved_evidence": retrieved_evidence,
        "disclaimer": "This tool is for educational purposes and is not medical advice. Always consult a certified physician for medical concerns."
    }


## Step 10 — Interactive Gradio Verification Interface
Interactive evaluation demo supporting English, Hindi, and Hinglish queries with preset test cases.


In [ ]:
def verify_pipeline(claim: str):
    if not claim or not claim.strip():
        return "N/A", "N/A", "Please enter a claim.", "N/A", "N/A", "", ""

    cleaned = preprocess_text(claim)
    lang = detect_language(claim)
    is_health = is_health_related(cleaned)

    if not is_health:
        v_data = {"verdict": "Non-Health Claim", "evidence_status": "Filtered", "risk": "None", "confidence": 0.0, "stance": "none"}
        exp = generate_explanation(claim, lang, False, v_data, [])
        return lang, "No (Filtered Out)", "Non-Health Claim", "None", exp["summary_text"], "No clinical retrieval needed.", exp["disclaimer"]

    retrieved = retrieve_evidence(cleaned, top_k=3)
    v_data = assess_verdict_and_risk(cleaned, retrieved)
    exp = generate_explanation(claim, lang, True, v_data, retrieved)

    evidence_md = "### 📚 Retrieved Clinical Evidence\n\n"
    for item in retrieved:
        evidence_md += (
            f"**Rank {item['rank']}** [Cosine Similarity: `{item['similarity_score']}` | Stance: `{item['stance'].upper()}`]\n"
            f"> *"{item['evidence_text']}"*\n"
            f"**Source**: {item['source']} — [Verified URL Link]({item['url']})\n\n"
            f"---\n"
        )

    return lang, "Yes (Passed Filter)", v_data["verdict"], v_data["risk"], exp["summary_text"], evidence_md, exp["disclaimer"]

with gr.Blocks(title="Healthcare Misinformation RAG") as demo:
    gr.Markdown("# 🩺 Multilingual Healthcare Misinformation Verifier (RAG)")
    gr.Markdown("Verify medical statements across **English**, **Hindi (हिंदी)**, and **Hinglish** using cross-lingual dense retrieval.")

    with gr.Row():
        with gr.Column(scale=2):
            input_box = gr.Textbox(label="Enter Claim", placeholder="e.g. Can masks reduce corona infections?", lines=2)
            verify_button = gr.Button("🔍 Verify Claim", variant="primary")
            gr.Markdown("### Demo Test Inputs")
            gr.Examples(
                examples=[
                    ["Can masks reduce corona infections when worn by a large proportion of the population?"],
                    ["Does arthroscopy help with osteoarthritis of the knee joint?"],
                    ["Can CBD help with migraines or other headaches?"],
                    ["Does paxlovid protect unvaccinated people with risk factors from severe covid?"],
                    ["Haldi doodh cancer ko theek karta hai"],
                    ["India won the cricket match"]
                ],
                inputs=[input_box]
            )

        with gr.Column(scale=2):
            with gr.Row():
                out_lang = gr.Textbox(label="Detected Language", interactive=False)
                out_health = gr.Textbox(label="Health-Related?", interactive=False)
            with gr.Row():
                out_verdict = gr.Textbox(label="Scientific Verdict", interactive=False)
                out_risk = gr.Textbox(label="Risk Assessment", interactive=False)

    with gr.Row():
        with gr.Column():
            out_explanation = gr.Markdown(label="Explanation")
            out_evidence = gr.Markdown(label="Retrieved Evidence")
            out_disclaimer = gr.Markdown(label="Disclaimer")

    verify_button.click(
        fn=verify_pipeline,
        inputs=[input_box],
        outputs=[out_lang, out_health, out_verdict, out_risk, out_explanation, out_evidence, out_disclaimer]
    )

# Launch Gradio interface (share=True creates a public Colab URL)
demo.launch(share=True, debug=False)
